In [ ]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset, DatasetInfo
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [ ]:
# full microbiology events table for modeling.  roughly 160k unique visits.  Includes
# cultures ordered for patients who eventually went to the hospital or were sent home.
# flagged results where results were known before they left ER (only 2%).  In most cases
# results weren't known so best to not include that in state

query_micro = """
SELECT
  e.subject_id,
  e.hadm_id,
  e.stay_id                    AS ed_stay_id,
  m.microevent_id,
  m.micro_specimen_id,
  m.charttime,
  m.spec_itemid,
  m.spec_type_desc,
  m.test_itemid,
  m.test_name,
  m.org_name,
  m.comments,
  m.storetime,
  IF(m.storetime IS NOT NULL AND m.storetime < e.outtime, 1, 0) AS result_available_in_er,
  e.disposition
FROM `physionet-data.mimiciv_ed.edstays` e
INNER JOIN `physionet-data.mimiciv_3_1_hosp.microbiologyevents` m
  ON e.subject_id = m.subject_id
WHERE m.charttime IS NOT NULL
  AND m.charttime BETWEEN e.intime AND e.outtime
  AND (
    e.hadm_id IS NOT NULL
    OR (e.hadm_id IS NULL AND e.disposition = 'HOME')
  )
ORDER BY e.subject_id, e.stay_id, m.charttime
"""

df_micro = client.query(query_micro_er).to_dataframe()

admitted = df_micro[df_micro['hadm_id'].notna()]
home     = df_micro[df_micro['hadm_id'].isna()]

print(f"Total rows          : {len(df_micro):,}")
print(f"Unique ED stays     : {df_micro['ed_stay_id'].nunique():,}")
print(f"Unique subjects     : {df_micro['subject_id'].nunique():,}")
print()
print(f"Admitted rows       : {len(admitted):,}  |  unique stays: {admitted['ed_stay_id'].nunique():,}")
print(f"Discharged home rows: {len(home):,}  |  unique stays: {home['ed_stay_id'].nunique():,}")
print()
# How many stays had more than one culture row?
rows_per_stay = df_micro.groupby('ed_stay_id').size()
print(f"Stays with 1 culture row  : {(rows_per_stay == 1).sum():,}")
print(f"Stays with 2+ culture rows: {(rows_per_stay > 1).sum():,}")
print(f"Max culture rows per stay : {rows_per_stay.max():,}")
print()
print(f"Result available in ER: {df_micro['result_available_in_er'].sum():,} ({df_micro['result_available_in_er'].mean()*100:.1f}%)")
df_micro.head(20)

In [ ]:
PHYSIONET_LICENSE = "PhysioNet Credentialed Health Data License 1.5.0 — https://physionet.org/content/mimiciv/view-license/3.1/"

info = DatasetInfo(
    description=(
        "Microbiology culture events from hosp.microbiologyevents for cohort patients during their stay window. "
        "Includes culture order time, specimen type, organism name, and antibiotic sensitivity results. "
        "culture_ordered is the real-time state signal; culture_positive is a retrospective label only "
        "(~2% of results available before ED discharge)."
    ),
    license=PHYSIONET_LICENSE,
)

ds = Dataset.from_pandas(df_micro, split='microbiology_events', info=info)
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="microbiology_events", data_dir="microbiology")
print("Pushed microbiology_events to HuggingFace Hub.")